# Session 3: Monte Carlo simulation, its error, and when it fails
**A. Real-world problem.** When counting is too hard we can *simulate*: play the game
thousands of times on a computer and count how often the event happens. How close is the
answer, how many plays do we need, and why can nobody simulate their way to a lottery jackpot?

*Education about chance only. The game and prizes are made-up; this is not betting advice.*

**Plan.** (1) estimate a probability by simulation, (2) measure the error with the standard
error, (3) look at the law of large numbers and the gambler's fallacy, (4) check a fair
game, (5) see the 6-of-49 lottery break simulation.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

LOTTERY_TOTAL = math.comb(49, 6)      # 13983816 equally likely draws

## Step 1: estimate by simulation and compare with the exact value
Monte Carlo idea: p_hat = (trials where the event happened) / N. A *seeded* generator
repeats exactly, so anyone can reproduce your number.

In [ ]:
def estimate_sum_probability(target, n, seed=1):
    """Estimate P(sum of two dice >= target) from n simulated rolls."""
    rng = np.random.default_rng(seed)
    s = rng.integers(1, 7, size=n) + rng.integers(1, 7, size=n)
    return float(np.mean(s >= target))


def exact_sum_at_least(target):
    """Exact P(sum of two dice >= target) by counting the 36 outcomes."""
    good = sum(1 for a in range(1, 7) for b in range(1, 7) if a + b >= target)
    return good / 36


def estimate_birthday(people, trials, seed=1):
    """Estimate P(at least two of `people` share a birthday) by simulation."""
    rng = np.random.default_rng(seed)
    days = np.sort(rng.integers(0, 365, size=(trials, people)), axis=1)
    return float(np.mean((np.diff(days, axis=1) == 0).any(axis=1)))

## Step 2: the error of a simulation
The estimate is itself random. For N independent trials of an event with probability p,
its typical error (the standard error) is

    SE = sqrt(p (1 - p) / N)

About 95 percent of estimates fall within 1.96 SE of the truth. To get a 95 percent half-width
of at most e, we need N >= 1.96^2 p (1 - p) / e^2. Ten times the accuracy costs 100 times the trials.

In [ ]:
def standard_error(p, n):
    return math.sqrt(p * (1 - p) / n)


def trials_needed(p, half_width, z=1.96):
    """Smallest N so that z * SE <= half_width."""
    if half_width <= 0:
        raise ValueError("half_width must be positive")
    return math.ceil(z * z * p * (1 - p) / half_width ** 2)


def running_estimate(target, n, seed=1):
    """Estimate after 1, 2, ..., n rolls (array of length n)."""
    rng = np.random.default_rng(seed)
    s = rng.integers(1, 7, size=n) + rng.integers(1, 7, size=n)
    return np.cumsum(s >= target) / np.arange(1, n + 1)

## Step 3: law of large numbers, and the gambler's fallacy
The law of large numbers says the *fraction* settles near p. It does not say the counts
even out: after a run of heads the next flip is still 50:50.

In [ ]:
def next_flip_after_streak(n, streak=5, seed=1):
    """Among n fair coin flips, the fraction of heads right after `streak` heads in a row."""
    rng = np.random.default_rng(seed)
    h = rng.integers(0, 2, size=n).astype(bool)
    run = np.ones(n - streak, dtype=bool)
    for k in range(streak):
        run &= h[k:n - streak + k]
    nxt = h[streak:]
    return float(nxt[run].mean()), int(run.sum())

## Step 4: a fair game, checked by simulation
Pay `price`, roll two dice, win a gross `payout` if the sum is at least 9 (probability 10/36).
Expected net gain = p x payout - price. It is 0 when payout = price / p.

In [ ]:
WIN_AT_LEAST = 9


def fair_payout(price, p):
    return price / p


def simulate_game(price, payout, n, seed=1):
    """Average net gain per play and its standard error over n simulated plays."""
    rng = np.random.default_rng(seed)
    s = rng.integers(1, 7, size=n) + rng.integers(1, 7, size=n)
    net = np.where(s >= WIN_AT_LEAST, payout - price, -price).astype(float)
    return float(net.mean()), float(net.std(ddof=1) / math.sqrt(n))

## Step 5: the lottery, and why simulation fails
A 6-of-49 ticket wins the jackpot with probability 1 / C(49, 6) = 1 / 13,983,816.
With an illustrative ticket price and jackpot the expected loss per ticket is
price - jackpot / 13983816 (smaller prizes ignored). To see a jackpot even once we
would need about 14 million simulated draws, and ten times that to estimate p to within 30 percent.

In [ ]:
def lottery_expected_loss(price, jackpot):
    """Expected loss per ticket if only the jackpot is counted."""
    return price - jackpot / LOTTERY_TOTAL


def simulate_lottery(draws, seed=1):
    """Number of simulated draws (out of `draws`) that equal the fixed ticket 1,2,3,4,5,6."""
    rng = np.random.default_rng(seed)
    hits, chunk = 0, 50000
    for start in range(0, draws, chunk):
        m = min(chunk, draws - start)
        picked = np.argpartition(rng.random((m, 49)), 6, axis=1)[:, :6] + 1
        hits += int(np.sum(np.all(np.sort(picked, axis=1) == np.arange(1, 7), axis=1)))
    return hits


def draws_for_relative_error(p, rel):
    """Trials so that the standard error is rel x p."""
    return math.ceil((1 - p) / (p * rel ** 2))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    p = exact_sum_at_least(9)
    print(f"exact P(sum >= 9) = 10/36 = {p:.4f}")
    for n in (100, 10000, 1000000):
        est = estimate_sum_probability(9, n)
        print(f"N = {n:7d}: estimate {est:.4f}, error {est - p:+.4f}, standard error {standard_error(p, n):.4f}")
    print("trials for 95% half-width 0.005 at p = 10/36:", trials_needed(p, 0.005))
    print(f"birthday, 23 people, 100000 trials: {estimate_birthday(23, 100000):.4f} (exact 0.5073)")
    frac, cnt = next_flip_after_streak(2000000)
    print(f"heads after 5 heads in a row: {frac:.4f} (from {cnt} streaks)")
    print("lottery total:", LOTTERY_TOTAL)
    print("expected loss on a Rs 100 ticket with a Rs 10 crore jackpot:", round(lottery_expected_loss(100, 1e8), 2))
    print("simulated jackpot hits in 200000 draws:", simulate_lottery(200000))
    print("draws needed for 30% relative error:", draws_for_relative_error(1 / LOTTERY_TOTAL, 0.3))

    # five seeded runs with the 95 percent error band
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ns = np.unique(np.logspace(1, 5, 40).astype(int))
    runs = np.array([[estimate_sum_probability(9, int(n), seed=s) for n in ns] for s in range(1, 6)])
    for r in runs:
        ax.semilogx(ns, r, alpha=0.6, lw=1)
    se = np.array([standard_error(p, n) for n in ns])
    ax.fill_between(ns, p - 1.96 * se, p + 1.96 * se, color="gray", alpha=0.2, label="exact +/- 1.96 SE")
    ax.axhline(p, color="red", ls="--", label="exact 10/36")
    ax.set_xlabel("number of simulated rolls N")
    ax.set_ylabel("estimate of P(sum >= 9)")
    ax.set_title("Five seeded runs inside the error band")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "error_band.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    pp = np.linspace(0.001, 1, 400)
    ax2.loglog(pp, [trials_needed(q, 0.01) for q in pp[:-1]] + [np.nan], label="half-width 0.01")
    ax2.loglog(pp, [trials_needed(q, 0.001) for q in pp[:-1]] + [np.nan], label="half-width 0.001")
    ax2.set_xlabel("true probability p")
    ax2.set_ylabel("trials N for 95 percent accuracy")
    ax2.set_title("Ten times the accuracy costs 100 times the trials")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "trials_needed.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint. Then try the mini-project in the README.

## Coding tasks
1. **Easy.** Use `estimate_sum_probability(9, N)` with N = 100, 10000 and 1000000. Predict the three errors from the standard error, then compare.
2. **Stretch.** Write `coverage(n, trials=500)` that counts how often the estimate of P(sum >= 9) lies within 1.96 standard errors of the exact value, for a given n, and print it for n = 50 and n = 5000.
3. **Explain.** Explain why the standard error formula would give 0 for a simulation that sees no jackpot, and why that is misleading.

## Assessment
- **A. Conceptual.** A coin has landed heads 5 times in a row. A friend says tails is now due. Another says the coin must be biased. Who is right?
- **B. Numerical.** You want to estimate P(two dice sum to at least 9) = 10/36 with a 95 percent half-width of 0.005. How many trials? How many if you want 0.0005?
- **C. Graph and simulation interpretation.** In the band figure, a run leaves the grey band at N = 200 but is inside it at N = 20000. Is the simulation broken?
- **D. Design and optimisation challenge.** Design a fair game: pay Rs 10, roll two dice, win a gross payout if the sum is at least 9. What payout makes it fair? Choose N to check it by simulation to within a standard error of 0.05 Rs and say what you would see for a payout of Rs 30.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def _raises(f):
    try:
        f()
    except ValueError:
        return True
    return False


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    p = L.exact_sum_at_least(9)
    checks.append(("exact P(sum >= 9) = 10/36", abs(p - 10 / 36) < 1e-12))
    est = L.estimate_sum_probability(9, 1000000, seed=1)
    checks.append(("1000000 simulated rolls are within 4 standard errors of 10/36", abs(est - p) < 4 * L.standard_error(p, 1000000)))
    checks.append(("same seed gives the same estimate", L.estimate_sum_probability(9, 5000, 4) == L.estimate_sum_probability(9, 5000, 4)))
    checks.append(("hand calculation: SE for p = 0.5, N = 100 is 0.05", abs(L.standard_error(0.5, 100) - 0.05) < 1e-12))
    checks.append(("worked example: p = 0.5 and half-width 0.01 need N = 9604 trials", L.trials_needed(0.5, 0.01) == 9604))
    checks.append(("ten times the accuracy needs about 100 times the trials",
                   abs(L.trials_needed(0.3, 0.001) / L.trials_needed(0.3, 0.01) - 100) < 0.1))
    errs = [abs(L.estimate_sum_probability(9, 400, seed=s) - p) for s in range(1, 201)]
    checks.append(("about 95 percent of N = 400 estimates fall within 1.96 SE", 0.90 <= np.mean(np.array(errs) <= 1.96 * L.standard_error(p, 400)) <= 0.99))
    checks.append(("birthday simulation for 23 people is close to 0.507", abs(L.estimate_birthday(23, 60000, seed=2) - 0.5073) < 0.01))
    frac, cnt = L.next_flip_after_streak(400000, 5, seed=3)
    checks.append(("gambler's fallacy: after 5 heads the next flip is still about 50 percent heads", cnt > 5000 and abs(frac - 0.5) < 0.02))
    checks.append(("lottery: C(49, 6) = 13983816 exactly", L.LOTTERY_TOTAL == 13983816 == math.comb(49, 6)))
    checks.append(("worked example: Rs 100 ticket, Rs 10 crore jackpot, expected loss Rs 92.85",
                   abs(L.lottery_expected_loss(100, 1e8) - 92.85) < 0.01))
    checks.append(("fair game: payout 36 for price 10 when p = 10/36", abs(L.fair_payout(10, p) - 36) < 1e-9))
    m, se = L.simulate_game(10, 36, 200000, seed=5)
    checks.append(("the fair game averages within 3 SE of 0", abs(m) < 3 * se))
    m2, se2 = L.simulate_game(10, 30, 200000, seed=5)
    checks.append(("an unfair payout of 30 shows a clear loss (expected -1.667)", m2 < -10 * se2 and abs(m2 + 1.667) < 0.2))
    checks.append(("failure case: 200000 simulated draws find no jackpot", L.simulate_lottery(100000, seed=1) == 0))
    checks.append(("failure case: 30 percent accuracy needs over 100 million draws", L.draws_for_relative_error(1 / L.LOTTERY_TOTAL, 0.3) > 1e8))
    checks.append(("edge case: zero half-width is rejected", _raises(lambda: L.trials_needed(0.5, 0))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
